# Generates ranked candidate panels for every figure slot

Writes `paper_figs_day2/*`, the source of most figure candidates in the paper (final picks are unrecorded -- see docs/INVENTORY.md section 0 item 8: `paper/` contains no image files, so the exact final choice can't be matched by checksum).

**Needs:** the rendered images from most of the notebooks above.

**Maintained runnable path:** none. This notebook is the only record of how figure candidates were assembled.

# Paper figure candidate factory

This notebook **does not choose final examples**. It generates many candidate panels with deterministic seeds and meaningful filenames so you can visually inspect them and choose the strongest examples for the paper.

It is built for the directory layout and CNT helpers used in `finish_day (1).ipynb`.

## Output
Everything is written under:

`/content/drive/MyDrive/DATA_ROOT/paper_figs_day2/`

Subfolders:

- `fig01_teaser/` — many full 3-row interpolation paths.
- `fig02_method/` — real-data method-figure candidates.
- `fig03_epsilon/` — plan-sharpness sweep candidates.
- `fig05_main_grid/` — many one-row method-comparison candidates + contact sheets.
- `fig06_element_zoom/` — automatically located local regions where linear and OT differ most.
- `fig07_boundary/` — correspondence-boundary candidates.
- `fig08_transfer/` — structure-preserving material-transfer candidates.
- `fig09_spatial_eta/` — spatially varying interpolation candidates.
- `fig10_palette/` — optional 4-way anchor-based palettes.
- `indexes/` — CSVs describing every candidate.

The first few cells use cached outputs only and are fast.  
The later cells initialize CNT and generate new outputs.

**Important:** selecting qualitative examples for a figure is fine, but if you choose them after inspection, describe them as *selected illustrative examples*, not *fixed before rendering*.

In [ ]:
# ============================================================
# 0. LIGHT SETUP — cached files only, no CNT model yet
# ============================================================
import os, re, glob, json, math, random, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFile
from google.colab import drive

drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True

BASE  = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE = f'{BASE}/tm_scale1000'
OUT   = f'{BASE}/matching_ablation'
PF    = f'{BASE}/paper_figs_day2'
IDX   = f'{PF}/indexes'

for d in [PF, IDX]:
    os.makedirs(d, exist_ok=True)

MAN_ALL = json.load(open(f'{SCALE}/manifest.json'))['pairs']
K_TO_P = {int(p['k']): p for p in MAN_ALL}

print('pairs:', len(MAN_ALL))
print('manifest example:', MAN_ALL[0])

DIRS = {
    'A':              f'{SCALE}/A',
    'B':              f'{SCALE}/B',
    'pixel_linear':   f'{SCALE}/pixel_linear',
    'texton_linear':  f'{SCALE}/texton_linear',
    'gaussian_w2':    f'{SCALE}/gaussian_w2',
    'ot':             f'{SCALE}/ot',
    'ot_sym':         f'{SCALE}/ot_sym',
    'hard':           f'{SCALE}/exact',
    'tm':             f'{SCALE}/mid',
    'vacher':         f'{SCALE}/vacher',
}

def slug(x, n=42):
    x = str(x)
    x = re.sub(r'[^A-Za-z0-9._-]+', '-', x).strip('-_')
    return (x[:n] or 'x')

def _walk_strings(obj, prefix=''):
    out = []
    if isinstance(obj, dict):
        for k,v in obj.items():
            out.extend(_walk_strings(v, f'{prefix}.{k}' if prefix else str(k)))
    elif isinstance(obj, (list, tuple)):
        for i,v in enumerate(obj):
            out.extend(_walk_strings(v, f'{prefix}[{i}]'))
    elif isinstance(obj, str):
        out.append((prefix, obj))
    return out

def endpoint_desc(k, side):
    p = K_TO_P[int(k)]
    side_l = side.lower()

    keys = [
        f'{side_l}_category', f'category_{side_l}', f'cat_{side_l}',
        f'{side_l}_class', f'class_{side_l}',
        f'{side_l}_name', f'name_{side_l}',
        f'{side}_category', f'{side}_name',
    ]
    for key in keys:
        if key in p and isinstance(p[key], (str,int,float)):
            return slug(p[key])

    for key in [side, side_l]:
        if key in p and isinstance(p[key], dict):
            q = p[key]
            for kk in ['category','cat','class','name','label','path','file']:
                if kk in q and isinstance(q[kk], str):
                    s = q[kk]
                    if '/' in s:
                        parts = Path(s).parts
                        if len(parts) >= 2:
                            return slug(parts[-2])
                    return slug(Path(s).stem)

    for key, val in _walk_strings(p):
        kl = key.lower()
        if side_l in kl and any(w in kl for w in ['path','file','image','img']):
            parts = Path(val).parts
            if len(parts) >= 2:
                return slug(parts[-2])
            return slug(Path(val).stem)

    return side

def pair_desc(k):
    return f'k{int(k):04d}__{endpoint_desc(k,"A")}_to_{endpoint_desc(k,"B")}'

def img_path(method, k):
    return f'{DIRS[method]}/{int(k):04d}.png'

def load_img(method, k, size=256):
    p = img_path(method, k)
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    return Image.open(p).convert('RGB').resize((size,size), Image.Resampling.LANCZOS)

def have(method, k):
    return os.path.exists(img_path(method,k))

def figdir(name):
    d = f'{PF}/{name}'
    os.makedirs(d, exist_ok=True)
    return d

def save_index(rows, name):
    p = f'{IDX}/{name}.csv'
    pd.DataFrame(rows).to_csv(p, index=False)
    print('index ->', p)
    return p

print('\nCached dirs:')
for k,p in DIRS.items():
    print(f'  {k:14s}', os.path.isdir(p), len(glob.glob(f'{p}/*.png')) if os.path.isdir(p) else 0)


In [ ]:
# ============================================================
# 1. BUILD CANDIDATE POOLS FROM CACHED FEATURES
# ============================================================
N_PER_POOL = 30
rng = np.random.RandomState(20260830)
ks = np.array([int(p['k']) for p in MAN_ALL])

def load_feature(method, observer='dino'):
    possibilities = [
        f'{OUT}/feat_{method}_{observer}.npy',
        f'{OUT}/feat_{method}_n1000_{observer}.npy',
    ]
    for p in possibilities:
        if os.path.exists(p):
            x = np.load(p)
            if len(x) >= len(ks):
                return x[:len(ks)]
    return None

F_OT  = load_feature('ot')
F_LIN = load_feature('texton_linear')
F_PIX = load_feature('pixel_linear')
F_G   = load_feature('gaussian_w2')
F_TM  = load_feature('tm')

ANC_PATH = f'{OUT}/anchor_dino.npz'
ANC = dict(np.load(ANC_PATH)) if os.path.exists(ANC_PATH) else None

metrics = pd.DataFrame({'k':ks})

if F_OT is not None and F_LIN is not None:
    metrics['d_ot_cntlin'] = np.linalg.norm(F_OT-F_LIN, axis=1)
if F_OT is not None and F_PIX is not None:
    metrics['d_ot_pixlin'] = np.linalg.norm(F_OT-F_PIX, axis=1)
if F_OT is not None and F_G is not None:
    metrics['d_ot_gauss'] = np.linalg.norm(F_OT-F_G, axis=1)
if F_OT is not None and F_TM is not None:
    metrics['d_ot_tm'] = np.linalg.norm(F_OT-F_TM, axis=1)

if ANC is not None:
    sep = []
    for k in ks:
        try:
            sep.append(np.linalg.norm(ANC[f'{k}_A']-ANC[f'{k}_B']))
        except KeyError:
            sep.append(np.nan)
    metrics['source_sep'] = sep

score_csv = f'{OUT}/final128_n1000.csv'
if os.path.exists(score_csv):
    sc = pd.read_csv(score_csv)
    try:
        piv = sc.pivot(index='k', columns='method', values='real_inc')
        if 'ot' in piv.columns and 'texton_linear' in piv.columns:
            adv = (piv['ot'] - piv['texton_linear']).rename('ot_inc_minus_cntlin')
            metrics = metrics.merge(adv, left_on='k', right_index=True, how='left')
    except Exception as e:
        print('score pivot skipped:', e)

POOLS = {}

def top_pool(col, n=N_PER_POOL, largest=True):
    if col not in metrics:
        return []
    z = metrics.dropna(subset=[col]).sort_values(col, ascending=not largest)
    return z.head(n).k.astype(int).tolist()

POOLS['random'] = rng.choice(ks, size=min(N_PER_POOL,len(ks)), replace=False).astype(int).tolist()
POOLS['high_OT_vs_CNTlinear_visual_gap'] = top_pool('d_ot_cntlin', largest=True)
POOLS['high_OT_vs_pixel_visual_gap']     = top_pool('d_ot_pixlin', largest=True)
POOLS['OT_Gaussian_close']               = top_pool('d_ot_gauss', largest=False)
POOLS['OT_Gaussian_different']           = top_pool('d_ot_gauss', largest=True)
POOLS['high_source_separation']          = top_pool('source_sep', largest=True)
POOLS['OT_realism_advantage']            = top_pool('ot_inc_minus_cntlin', largest=True)

if all(x is not None for x in [F_OT,F_LIN,F_G,F_TM]):
    allclose = (
        np.linalg.norm(F_OT-F_LIN,axis=1) +
        np.linalg.norm(F_OT-F_G,axis=1) +
        np.linalg.norm(F_OT-F_TM,axis=1)
    )
    metrics['all_methods_spread'] = allclose
    POOLS['all_methods_close'] = metrics.sort_values('all_methods_spread').head(N_PER_POOL).k.astype(int).tolist()

required = ['A','B','texton_linear','gaussian_w2','ot','tm']
for name in list(POOLS):
    POOLS[name] = [
        k for k in POOLS[name]
        if all(have(m,k) for m in required)
    ]

rows=[]
for pool, arr in POOLS.items():
    for rank,k in enumerate(arr,1):
        r={'pool':pool,'rank':rank,'k':k,'pair':pair_desc(k)}
        mm = metrics.loc[metrics.k==k]
        if len(mm):
            r.update(mm.iloc[0].to_dict())
        rows.append(r)

save_index(rows, 'candidate_pools')
display(pd.DataFrame(rows).head(30))
print('\nPools:')
for k,v in POOLS.items():
    print(f'{k:34s} {len(v)}')


In [ ]:
# ============================================================
# 2. FIGURE 5 — MANY MAIN-COMPARISON ROWS
# ============================================================
D = figdir('fig05_main_grid')
MAX_PER_POOL = 24

base_cols = [
    ('A', 'A'),
    ('CNT-linear', 'texton_linear'),
    (r'Gaussian $W_2$', 'gaussian_w2'),
    ('OT', 'ot'),
]
if os.path.isdir(DIRS['vacher']) and len(glob.glob(f"{DIRS['vacher']}/*.png")):
    base_cols.append((r'Vacher-$W_2$', 'vacher'))
base_cols += [
    ('Texture Mixer', 'tm'),
    ('B', 'B')
]

saved=[]
seen=set()

for pool, arr in POOLS.items():
    for rank,k in enumerate(arr[:MAX_PER_POOL],1):
        key=(pool,k)
        if key in seen:
            continue
        seen.add(key)

        cols=[c for c in base_cols if have(c[1],k)]
        if len(cols) < 6:
            continue

        fig, ax = plt.subplots(1, len(cols), figsize=(2.05*len(cols),2.35), dpi=160)
        if len(cols)==1:
            ax=[ax]
        for j,(title,m) in enumerate(cols):
            ax[j].imshow(load_img(m,k,256))
            ax[j].axis('off')
            ax[j].set_title(title, fontsize=9)

        fname = f'fig05__{slug(pool)}__r{rank:02d}__{pair_desc(k)}.png'
        fp = f'{D}/{fname}'
        plt.tight_layout(pad=.5)
        plt.savefig(fp, dpi=220, bbox_inches='tight')
        plt.close(fig)
        saved.append(dict(figure='fig05',pool=pool,rank=rank,k=k,file=fp,pair=pair_desc(k)))

print('saved',len(saved),'main-grid row candidates ->',D)
save_index(saved,'fig05_main_grid')


In [ ]:
# ============================================================
# 3. FIGURE 6 — AUTOMATIC ELEMENT-ZOOM CANDIDATES
# ============================================================
import cv2

D = figdir('fig06_element_zoom')
CROP = 80
N_CAND = 35

def max_diff_crop(im1, im2, crop=CROP):
    a=np.asarray(im1.resize((256,256))).astype(np.float32)/255.
    b=np.asarray(im2.resize((256,256))).astype(np.float32)/255.
    diff=np.abs(a-b).mean(2)
    score=cv2.boxFilter(diff, ddepth=-1, ksize=(crop,crop), normalize=True,
                        borderType=cv2.BORDER_REFLECT)
    cy,cx=np.unravel_index(np.argmax(score),score.shape)
    x=max(0,min(256-crop,int(cx-crop//2)))
    y=max(0,min(256-crop,int(cy-crop//2)))
    return x,y,float(score[cy,cx])

def boxed(im, xy, color=(230,30,30), width=4):
    z=im.copy()
    d=ImageDraw.Draw(z)
    x,y,w,h=xy
    d.rectangle((x,y,x+w,y+h), outline=color, width=width)
    return z

cand=[]
for col in ['d_ot_pixlin','d_ot_cntlin']:
    if col in metrics:
        cand += metrics.sort_values(col,ascending=False).head(N_CAND).k.astype(int).tolist()
cand += POOLS.get('random',[])[:15]
cand=list(dict.fromkeys(cand))

saved=[]
for baseline in ['pixel_linear','texton_linear']:
    nice_name = 'pixel-linear' if baseline=='pixel_linear' else 'CNT-linear'
    count=0
    for rank,k in enumerate(cand,1):
        if not all(have(m,k) for m in ['A','B',baseline,'ot']):
            continue

        A=load_img('A',k); B=load_img('B',k)
        L=load_img(baseline,k); O=load_img('ot',k)
        x,y,s=max_diff_crop(L,O,CROP)

        full=[A,boxed(L,(x,y,CROP,CROP)),boxed(O,(x,y,CROP,CROP)),B]
        crops=[None,L.crop((x,y,x+CROP,y+CROP)),O.crop((x,y,x+CROP,y+CROP)),None]

        fig,ax=plt.subplots(2,4,figsize=(9.0,4.9),dpi=160,
                            gridspec_kw={'height_ratios':[1,1]})
        titles=['A',nice_name,'OT','B']
        for j,im in enumerate(full):
            ax[0,j].imshow(im)
            ax[0,j].axis('off')
            ax[0,j].set_title(titles[j],fontsize=9)

        for j in range(4):
            ax[1,j].axis('off')
            if crops[j] is not None:
                ax[1,j].imshow(crops[j],interpolation='nearest')
        ax[1,1].set_title(f'crop ({x},{y})',fontsize=8)
        ax[1,2].set_title(f'local MAD score {s:.3f}',fontsize=8)

        fname=f'fig06__{slug(nice_name)}__rank{rank:02d}__{pair_desc(k)}__crop_x{x}_y{y}.png'
        fp=f'{D}/{fname}'
        plt.tight_layout(pad=.45)
        plt.savefig(fp,dpi=240,bbox_inches='tight')
        plt.close(fig)

        saved.append(dict(figure='fig06',baseline=nice_name,rank=rank,k=k,
                          x=x,y=y,local_diff=s,file=fp,pair=pair_desc(k)))
        count += 1
        if count>=N_CAND:
            break

print('saved',len(saved),'element-zoom candidates ->',D)
save_index(saved,'fig06_element_zoom')


# CNT-dependent figures

The remaining cells generate new outputs, so they initialize CNT.  
They do **not** rescore the paper or overwrite the canonical quantitative caches.

In [ ]:
# ============================================================
# HOTFIX — install Detectron2 before CNT model construction
# ============================================================
import sys
import subprocess
import importlib

try:
    import detectron2
    print(
        "detectron2 already installed:",
        getattr(detectron2, "__version__", "unknown")
    )

except Exception:
    print("Installing detectron2...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/facebookresearch/detectron2.git"
    ])

    importlib.invalidate_caches()

    import detectron2

    print(
        "detectron2 installed:",
        getattr(detectron2, "__version__", "unknown")
    )

In [ ]:
# ============================================================
# 4. CNT MODEL SETUP — no DTD download, no observers
# ============================================================
import os, sys, time, gc
from pathlib import Path
import torch
import torch.nn.functional as F

device='cuda'
TMP='/content/tmp_imgs_figday'
os.makedirs(TMP,exist_ok=True)

os.system('pip install -q einops hydra-core omegaconf 2>/dev/null')

if 'model' not in globals():
    os.chdir('/content')
    if not os.path.isdir('/content/cnt-siga24'):
        os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
    CNT='/content/cnt-siga24'
    os.system(f'rm -rf "{CNT}/ckpts"; ln -sfn "{BASE}/cnt_ckpts" "{CNT}/ckpts"')
    os.chdir(CNT)
    if CNT not in sys.path:
        sys.path.insert(0,CNT)
    try:
        from src.inference.core import TexAutoEncoderInference, load_image
    except Exception:
        print('Installing CNT requirements...')
        os.system('pip install -q -r requirements.in')
        try:
            from src.inference.core import TexAutoEncoderInference, load_image
        except Exception:
            os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
            from src.inference.core import TexAutoEncoderInference, load_image

    model=TexAutoEncoderInference(
        ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),
        device=device
    )

IMG_SIZE=256
FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)):
        return load_image(Path(x),size).to(device)
    _tc[0]+=1
    fp=f'{TMP}/t{_tc[0]%64}.png'
    x.convert('RGB').save(fp)
    return load_image(Path(fp),size).to(device)

def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))

def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img)
        L=model.splat(blob)
        return (
            blob,
            L["layer_level_layouts"][-1][0]["blob_features"].clone(),
            L["layer_level_layouts"][-1][0]["score_img"].clone()
        )

def build_style_maps(bf,si,fms):
    sp={}
    s=si
    sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False)
        sp[s.shape[-1]]=s
    return {
        size:torch.einsum("bnc,bnhw->bchw",bf,sp[size])
        for size in set(fms)
    }

def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref)
        si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}
        s=si
        sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False)
            sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(
            1,model.generator.n_embedding,1,1
        )
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1:
                break
            elif fms[i]!=fms[i+1]:
                x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)

def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia)
    bb,bfb,sib=get_gen_inputs(ib)
    return {
        "blob_a":ba,"bf_a":bfa,"si_a":sia,
        "blob_b":bb,"bf_b":bfb,"si_b":sib
    }

def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS)
    sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS)
    A,B=sa[L][0],sb[L][0]
    Cc,H,W=A.shape
    return (
        L,Cc,H,W,
        A.permute(1,2,0).reshape(-1,Cc),
        B.permute(1,2,0).reshape(-1,Cc)
    )

_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag])
    return torch.randperm(N,generator=g,device=device)[:NSUB]

def _dec(d,it,L,H,W,Cc,layout='a'):
    it=it.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    maps={
        s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False))
        for s in set(FMS)
    }
    return decode_with_style_maps(d["blob_"+layout],maps)

def interp_lin_pixel(d,eta):
    # Style-map positional interpolation.
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS)
    sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(
        d["blob_a"],
        {s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)}
    )

def interp_lin_texton(d,eta):
    bf=(1-eta)*d["bf_a"]+eta*d["bf_b"]
    si=(1-eta)*d["si_a"]+eta*d["si_b"]
    sm=build_style_maps(bf,si,FMS)
    return decode_with_style_maps(d["blob_a"],sm)

def sinkhorn_plan(x,y,eps,iters=ITERS):
    C=torch.cdist(x,y).pow(2)
    K=torch.exp(-C/eps)
    u=torch.ones(len(x),device=x.device)/len(x)
    v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u)
    b=torch.ones_like(v)
    for _ in range(iters):
        a=u/(K@b+1e-30)
        b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:],C

@torch.no_grad()
def transport_field(d,seed=0,eps=EPS):
    L,Cc,H,W,sf,tf=_grids(d)
    N=sf.shape[0]
    i_s=_sub(N,seed,'s')
    i_t=_sub(N,seed,'t')
    X=sf[i_s]
    Y=tf[i_t]
    P,C=sinkhorn_plan(X,Y,eps)
    Q=P/(P.sum(1,keepdim=True)+1e-10)
    tt=Q@Y
    nn=torch.cdist(sf,X).argmin(1)
    return {
        'L':L,'Cc':Cc,'H':H,'W':W,'sf':sf,'tf':tf,
        'i_s':i_s,'i_t':i_t,'X':X,'Y':Y,
        'P':P,'Q':Q,'C':C,'tt':tt,'target_full':tt[nn]
    }

@torch.no_grad()
def decode_field(d,fld,eta):
    x=(1-eta)*fld['sf']+eta*fld['target_full']
    return _dec(d,x,fld['L'],fld['H'],fld['W'],fld['Cc'])

def pair_imgs(k):
    return load_img('A',k,256),load_img('B',k,256)

print('CNT figure-generation helpers ready.')


In [ ]:
# ============================================================
# 5. FIGURE 1 — MANY FULL TEASER PATHS
# ============================================================
D=figdir('fig01_teaser')
ETAS=np.linspace(0,1,8)

TEASER_KS=[]
for name in ['high_OT_vs_CNTlinear_visual_gap','high_source_separation','random']:
    TEASER_KS += POOLS.get(name,[])[:12]
TEASER_KS=list(dict.fromkeys(TEASER_KS))[:30]

saved=[]
for rank,k in enumerate(TEASER_KS,1):
    A,B=pair_imgs(k)
    d=build_d(cnt_load(A),cnt_load(B))
    fld=transport_field(d,seed=k,eps=EPS)

    alpha=[Image.blend(A,B,float(e)) for e in ETAS]
    linear=[to_pil(interp_lin_pixel(d,float(e))) for e in ETAS]
    ot=[to_pil(decode_field(d,fld,float(e))) for e in ETAS]

    fig,ax=plt.subplots(3,len(ETAS),figsize=(15.8,5.8),dpi=150)
    for i,row in enumerate([alpha,linear,ot]):
        for j,im in enumerate(row):
            ax[i,j].imshow(im)
            ax[i,j].axis('off')
            if i==0:
                ax[i,j].set_title(rf'$\eta={ETAS[j]:.2f}$',fontsize=8)
    for i,label in enumerate(['pixel cross-fade','CNT-linear','OT (ours)']):
        ax[i,0].set_ylabel(label,fontsize=9,rotation=90,labelpad=8)

    fname=f'fig01__teaser__rank{rank:02d}__{pair_desc(k)}.png'
    fp=f'{D}/{fname}'
    plt.tight_layout(pad=.25)
    plt.savefig(fp,dpi=220,bbox_inches='tight')
    plt.close(fig)
    saved.append(dict(rank=rank,k=k,file=fp,pair=pair_desc(k)))

    del d,fld
    if rank%5==0:
        gc.collect()
        torch.cuda.empty_cache()
        print(rank,'/',len(TEASER_KS))

print('saved',len(saved),'teaser candidates ->',D)
save_index(saved,'fig01_teaser')


In [ ]:
# ============================================================
# 6. FIGURE 2 — REAL-DATA METHOD FIGURE CANDIDATES
# ============================================================
from sklearn.decomposition import PCA

D=figdir('fig02_method')

METHOD_KS=[]
for name in ['high_OT_vs_CNTlinear_visual_gap','random']:
    METHOD_KS += POOLS.get(name,[])[:10]
METHOD_KS=list(dict.fromkeys(METHOD_KS))[:16]

def pca_rgb_pair(sf,tf,seed=0):
    rng=np.random.RandomState(seed)
    n=min(4096,len(sf))
    idx=rng.choice(len(sf),n,replace=False)
    fit=np.concatenate([
        sf[idx].detach().cpu().numpy(),
        tf[idx].detach().cpu().numpy()
    ],axis=0)
    p=PCA(3,random_state=seed).fit(fit)
    A3=p.transform(sf.detach().cpu().numpy())
    B3=p.transform(tf.detach().cpu().numpy())
    both=np.concatenate([A3,B3],axis=0)
    lo=np.percentile(both,1,axis=0)
    hi=np.percentile(both,99,axis=0)
    def norm(z):
        return np.clip((z-lo)/(hi-lo+1e-9),0,1)
    H=int(round(math.sqrt(len(sf))))
    return norm(A3).reshape(H,H,3), norm(B3).reshape(H,H,3)

saved=[]
for rank,k in enumerate(METHOD_KS,1):
    A,B=pair_imgs(k)
    d=build_d(cnt_load(A),cnt_load(B))
    fld=transport_field(d,seed=k,eps=EPS)
    sf,tf=fld['sf'],fld['tf']

    smA,smB=pca_rgb_pair(sf,tf,seed=k)

    fit=np.concatenate([
        fld['X'].detach().cpu().numpy(),
        fld['Y'].detach().cpu().numpy()
    ],axis=0)
    p2=PCA(2,random_state=k).fit(fit)

    rng=np.random.RandomState(k+17)
    pos_idx=rng.choice(len(sf),22,replace=False)
    posA=p2.transform(sf[pos_idx].detach().cpu().numpy())
    posB=p2.transform(tf[pos_idx].detach().cpu().numpy())

    choose=np.arange(min(22,len(fld['X'])))
    otA=p2.transform(fld['X'][choose].detach().cpu().numpy())
    otT=p2.transform(fld['tt'][choose].detach().cpu().numpy())
    target_cloud=p2.transform(
        fld['Y'][::max(1,len(fld["Y"])//250)].detach().cpu().numpy()
    )

    cfig,cax=plt.subplots(1,2,figsize=(7.2,3.2),dpi=160)
    cax[0].scatter(posA[:,0],posA[:,1],s=18,label='A features')
    cax[0].scatter(posB[:,0],posB[:,1],s=18,label='B features')
    for i in range(len(posA)):
        cax[0].plot([posA[i,0],posB[i,0]],[posA[i,1],posB[i,1]],lw=.75,alpha=.45)
    cax[0].set_title('positional pairing')
    cax[0].legend(fontsize=7,loc='best')

    cax[1].scatter(target_cloud[:,0],target_cloud[:,1],s=5,alpha=.22,label='target cloud')
    cax[1].scatter(otA[:,0],otA[:,1],s=18,label='source')
    cax[1].scatter(otT[:,0],otT[:,1],s=18,marker='x',label='OT destination')
    for i in range(len(otA)):
        cax[1].annotate('',xy=otT[i],xytext=otA[i],
                        arrowprops=dict(arrowstyle='->',lw=.8,alpha=.6))
    cax[1].set_title('OT barycentric transport')
    cax[1].legend(fontsize=7,loc='best')
    for a in cax:
        a.set_xticks([])
        a.set_yticks([])
    cfig.tight_layout()
    coupling_fp=f'{D}/fig02__coupling__rank{rank:02d}__{pair_desc(k)}.png'
    cfig.savefig(coupling_fp,dpi=230,bbox_inches='tight')
    plt.close(cfig)

    e5=np.linspace(0,1,5)
    paths=[to_pil(decode_field(d,fld,float(e))) for e in e5]
    tfig,tax=plt.subplots(1,5,figsize=(9.3,2.0),dpi=160)
    for j,im in enumerate(paths):
        tax[j].imshow(im)
        tax[j].axis('off')
        tax[j].set_title(rf'$\eta={e5[j]:.2f}$',fontsize=8)
    tfig.tight_layout(pad=.25)
    transition_fp=f'{D}/fig02__transition__rank{rank:02d}__{pair_desc(k)}.png'
    tfig.savefig(transition_fp,dpi=230,bbox_inches='tight')
    plt.close(tfig)

    fig=plt.figure(figsize=(15.8,6.0),dpi=150)
    gs=fig.add_gridspec(2,9,width_ratios=[1,1,1.7,1.7,1,1,1,1,1])

    ax=fig.add_subplot(gs[0,0]); ax.imshow(A); ax.set_title('Texture A',fontsize=9); ax.axis('off')
    ax=fig.add_subplot(gs[1,0]); ax.imshow(B); ax.set_title('Texture B',fontsize=9); ax.axis('off')
    ax=fig.add_subplot(gs[0,1]); ax.imshow(smA); ax.set_title('style map A\n(PCA→RGB)',fontsize=8); ax.axis('off')
    ax=fig.add_subplot(gs[1,1]); ax.imshow(smB); ax.set_title('style map B\n(PCA→RGB)',fontsize=8); ax.axis('off')

    ax=fig.add_subplot(gs[:,2])
    ax.scatter(posA[:,0],posA[:,1],s=16)
    ax.scatter(posB[:,0],posB[:,1],s=16)
    for i in range(len(posA)):
        ax.plot([posA[i,0],posB[i,0]],[posA[i,1],posB[i,1]],lw=.65,alpha=.4)
    ax.set_title('positional\npairing',fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

    ax=fig.add_subplot(gs[:,3])
    ax.scatter(target_cloud[:,0],target_cloud[:,1],s=4,alpha=.18)
    ax.scatter(otA[:,0],otA[:,1],s=16)
    ax.scatter(otT[:,0],otT[:,1],s=16,marker='x')
    for i in range(len(otA)):
        ax.annotate('',xy=otT[i],xytext=otA[i],
                    arrowprops=dict(arrowstyle='->',lw=.7,alpha=.55))
    ax.set_title('appearance-aware\nOT transport',fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

    for j,im in enumerate(paths):
        ax=fig.add_subplot(gs[:,4+j])
        ax.imshow(im)
        ax.axis('off')
        ax.set_title(rf'$\eta={e5[j]:.2f}$',fontsize=8)

    fname=f'fig02__method__rank{rank:02d}__{pair_desc(k)}.png'
    fp=f'{D}/{fname}'
    fig.tight_layout(pad=.45)
    fig.savefig(fp,dpi=220,bbox_inches='tight')
    plt.close(fig)

    saved.append(dict(rank=rank,k=k,file=fp,coupling=coupling_fp,
                      transition=transition_fp,pair=pair_desc(k)))

    del d,fld
    gc.collect()
    torch.cuda.empty_cache()
    print(rank,'/',len(METHOD_KS),pair_desc(k))

save_index(saved,'fig02_method')
print('method candidates ->',D)


In [ ]:
# ============================================================
# 7. FIGURE 3 — EPSILON / PLAN-SHARPNESS SWEEP CANDIDATES
# ============================================================
D=figdir('fig03_epsilon')
EPS_LIST=[0.005,0.01,0.02,0.05,0.10,0.20,0.50]

EPS_KS=[]
for name in ['high_OT_vs_CNTlinear_visual_gap','random']:
    EPS_KS += POOLS.get(name,[])[:7]
EPS_KS=list(dict.fromkeys(EPS_KS))[:12]

saved=[]
for rank,k in enumerate(EPS_KS,1):
    A,B=pair_imgs(k)
    d=build_d(cnt_load(A),cnt_load(B))

    outs=[]
    labels=[]
    for eps in EPS_LIST:
        fld=transport_field(d,seed=k,eps=eps)
        out=to_pil(decode_field(d,fld,.5))

        Q=fld['Q']
        ent=float((-(Q*(Q+1e-30).log()).sum(1)).mean().detach().cpu())
        tvar=float(fld['tt'].var(0,unbiased=False).sum().detach().cpu())
        yvar=float(fld['Y'].var(0,unbiased=False).sum().detach().cpu())
        vr=tvar/(yvar+1e-12)

        outs.append(out)
        labels.append((eps,ent,vr))
        del fld

    fig,ax=plt.subplots(1,len(EPS_LIST)+2,figsize=(17.0,2.7),dpi=150)
    ax[0].imshow(A)
    ax[0].set_title('A',fontsize=9)
    ax[0].axis('off')
    for j,(im,(eps,ent,vr)) in enumerate(zip(outs,labels),1):
        ax[j].imshow(im)
        ax[j].axis('off')
        ax[j].set_title(
            rf'$\epsilon={eps:g}$'+'\n'+f'H={ent:.2f}, var={100*vr:.0f}%',
            fontsize=7.5
        )
    ax[-1].imshow(B)
    ax[-1].set_title('B',fontsize=9)
    ax[-1].axis('off')

    fname=f'fig03__epsilon_sweep__rank{rank:02d}__{pair_desc(k)}.png'
    fp=f'{D}/{fname}'
    plt.tight_layout(pad=.35)
    plt.savefig(fp,dpi=220,bbox_inches='tight')
    plt.close(fig)

    saved.append(dict(rank=rank,k=k,file=fp,pair=pair_desc(k),
                      eps_values=';'.join(map(str,EPS_LIST))))
    del d
    gc.collect()
    torch.cuda.empty_cache()
    print(rank,'/',len(EPS_KS))

save_index(saved,'fig03_epsilon')
print('epsilon candidates ->',D)


In [ ]:
# ============================================================
# 8. FIGURE 7 — CORRESPONDENCE-BOUNDARY CANDIDATES
# ============================================================
import cv2

D=figdir('fig07_boundary')

def rotate_reflect(pil,deg):
    a=np.asarray(pil.convert('RGB'))
    h,w=a.shape[:2]
    M=cv2.getRotationMatrix2D((w/2,h/2),deg,1.0)
    z=cv2.warpAffine(a,M,(w,h),flags=cv2.INTER_CUBIC,borderMode=cv2.BORDER_REFLECT_101)
    return Image.fromarray(z)

def shifted_crops(pil,shift=32,crop=224):
    p=pil.resize((256,256),Image.Resampling.LANCZOS)
    a=p.crop((0,0,crop,crop)).resize((256,256),Image.Resampling.LANCZOS)
    b=p.crop((shift,shift,shift+crop,shift+crop)).resize((256,256),Image.Resampling.LANCZOS)
    return a,b

def sift_ransac_inliers(a,b):
    g=lambda im: cv2.cvtColor(np.asarray(im.resize((256,256))),cv2.COLOR_RGB2GRAY)
    sift=cv2.SIFT_create()
    k1,d1=sift.detectAndCompute(g(a),None)
    k2,d2=sift.detectAndCompute(g(b),None)
    if d1 is None or d2 is None or len(k1)<4 or len(k2)<4:
        return 0,0
    raw=cv2.BFMatcher().knnMatch(d1,d2,k=2)
    good=[m for m,n in raw if m.distance < .75*n.distance]
    if len(good)<4:
        return len(good),0
    src=np.float32([k1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst=np.float32([k2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H,mask=cv2.findHomography(src,dst,cv2.RANSAC,5.0)
    return len(good), int(mask.sum()) if mask is not None else 0

BOUND_KS=[]
for name in ['random','high_OT_vs_pixel_visual_gap']:
    BOUND_KS += POOLS.get(name,[])[:12]
BOUND_KS=list(dict.fromkeys(BOUND_KS))[:20]

saved=[]
for rank,k in enumerate(BOUND_KS,1):
    A,B=pair_imgs(k)
    c1,c2=shifted_crops(A,shift=32,crop=224)
    cases=[
        (c1,c2,'overlap-crops'),
        (A,rotate_reflect(A,2),'warp-02deg'),
        (A,rotate_reflect(A,8),'warp-08deg'),
        (A,B,'unrelated'),
    ]

    rendered=[]
    for ci,(aa,bb,tag) in enumerate(cases):
        d=build_d(cnt_load(aa),cnt_load(bb))
        lin=to_pil(interp_lin_pixel(d,.5))
        fld=transport_field(d,seed=k*10+ci,eps=EPS)
        ot=to_pil(decode_field(d,fld,.5))
        good,inl=sift_ransac_inliers(aa,bb)
        rendered.append((tag,good,inl,[aa,lin,ot,bb]))
        del d,fld

    fig,ax=plt.subplots(len(rendered),4,figsize=(9.4,9.0),dpi=150)
    for i,(tag,good,inl,ims) in enumerate(rendered):
        for j,im in enumerate(ims):
            ax[i,j].imshow(im.resize((256,256)))
            ax[i,j].axis('off')
        ax[i,0].set_ylabel(
            f'{tag}\nSIFT {inl}/{good} inliers',
            fontsize=8,rotation=0,ha='right',va='center',labelpad=66
        )
        if i==0:
            for j,t in enumerate(['A','positional linear','OT','B']):
                ax[i,j].set_title(t,fontsize=9)

        rowfig,rowax=plt.subplots(1,4,figsize=(8.8,2.15),dpi=150)
        for j,im in enumerate(ims):
            rowax[j].imshow(im.resize((256,256)))
            rowax[j].axis('off')
            rowax[j].set_title(['A','linear','OT','B'][j],fontsize=8)
        rowfig.suptitle(f'{tag} — SIFT RANSAC inliers {inl}/{good}',fontsize=9)
        rowfig.tight_layout(pad=.35)
        rowfp=f'{D}/row__{tag}__rank{rank:02d}__{pair_desc(k)}__sift{inl}-of-{good}.png'
        rowfig.savefig(rowfp,dpi=220,bbox_inches='tight')
        plt.close(rowfig)

    fname=f'fig07__boundary__rank{rank:02d}__{pair_desc(k)}.png'
    fp=f'{D}/{fname}'
    plt.tight_layout(pad=.4)
    plt.savefig(fp,dpi=220,bbox_inches='tight')
    plt.close(fig)

    saved.append(dict(rank=rank,k=k,file=fp,pair=pair_desc(k)))
    gc.collect()
    torch.cuda.empty_cache()
    print(rank,'/',len(BOUND_KS))

save_index(saved,'fig07_boundary')
print('boundary candidates ->',D)


In [ ]:
# ============================================================
# 9. FIGURE 8 — MANY STRUCTURE-PRESERVING MATERIAL TRANSFERS
# ============================================================
D=figdir('fig08_transfer')

TRANSFER_KS=[]
for name in ['high_source_separation','high_OT_vs_CNTlinear_visual_gap','random']:
    TRANSFER_KS += POOLS.get(name,[])[:18]
TRANSFER_KS=list(dict.fromkeys(TRANSFER_KS))[:45]

saved=[]
for rank,k in enumerate(TRANSFER_KS,1):
    A,B=pair_imgs(k)

    dab=build_d(cnt_load(A),cnt_load(B))
    fab=transport_field(dab,seed=k,eps=EPS)
    AB=to_pil(decode_field(dab,fab,1.0))

    dba=build_d(cnt_load(B),cnt_load(A))
    fba=transport_field(dba,seed=k,eps=EPS)
    BA=to_pil(decode_field(dba,fba,1.0))

    fig,ax=plt.subplots(1,4,figsize=(9.2,2.45),dpi=160)
    ims=[A,AB,BA,B]
    titles=['A','A layout + B material','B layout + A material','B']
    for j,(im,t) in enumerate(zip(ims,titles)):
        ax[j].imshow(im)
        ax[j].axis('off')
        ax[j].set_title(t,fontsize=8.5)

    fname=f'fig08__transfer__rank{rank:02d}__{pair_desc(k)}.png'
    fp=f'{D}/{fname}'
    plt.tight_layout(pad=.45)
    plt.savefig(fp,dpi=230,bbox_inches='tight')
    plt.close(fig)

    AB.save(f'{D}/raw__{pair_desc(k)}__A-layout_B-material.png')
    BA.save(f'{D}/raw__{pair_desc(k)}__B-layout_A-material.png')

    saved.append(dict(rank=rank,k=k,file=fp,pair=pair_desc(k)))
    del dab,dba,fab,fba
    if rank%5==0:
        gc.collect()
        torch.cuda.empty_cache()
        print(rank,'/',len(TRANSFER_KS))

save_index(saved,'fig08_transfer')
print('transfer candidates ->',D)


In [ ]:
# ============================================================
# 10. FIGURE 9 — SPATIALLY VARYING eta(x) CANDIDATES
# ============================================================
D=figdir('fig09_spatial_eta')
SPATIAL_KS=list(dict.fromkeys(
    POOLS.get('high_OT_vs_CNTlinear_visual_gap',[])[:10] +
    POOLS.get('random',[])[:10]
))[:16]

def eta_fields(H,W,device):
    yy,xx=torch.meshgrid(
        torch.linspace(0,1,H,device=device),
        torch.linspace(0,1,W,device=device),
        indexing='ij'
    )
    horizontal=xx
    vertical=yy
    rr=torch.sqrt((xx-.5)**2+(yy-.5)**2)/math.sqrt(.5)
    radial=torch.clamp(rr,0,1)
    soft_disk=torch.sigmoid((0.34-rr)*22.0)
    return {
        'horizontal ramp':horizontal,
        'vertical ramp':vertical,
        'radial ramp':radial,
        'soft disk':soft_disk,
    }

saved=[]
for rank,k in enumerate(SPATIAL_KS,1):
    A,B=pair_imgs(k)
    d=build_d(cnt_load(A),cnt_load(B))
    fld=transport_field(d,seed=k,eps=EPS)
    fields=eta_fields(fld['H'],fld['W'],device)

    fig,ax=plt.subplots(len(fields),4,figsize=(8.7,2.05*len(fields)),dpi=150)
    for i,(name,m) in enumerate(fields.items()):
        mv=m.reshape(-1,1)
        style=(1-mv)*fld['sf'] + mv*fld['target_full']
        out=to_pil(_dec(d,style,fld['L'],fld['H'],fld['W'],fld['Cc']))

        ax[i,0].imshow(A)
        ax[i,0].axis('off')
        ax[i,1].imshow(m.detach().cpu().numpy(),cmap='gray',vmin=0,vmax=1)
        ax[i,1].axis('off')
        ax[i,2].imshow(out)
        ax[i,2].axis('off')
        ax[i,3].imshow(B)
        ax[i,3].axis('off')
        ax[i,0].set_ylabel(name,fontsize=8,rotation=0,ha='right',va='center',labelpad=55)
        if i==0:
            for j,t in enumerate(['A',r'$\eta(x)$','output','B']):
                ax[i,j].set_title(t,fontsize=9)

        out.save(f'{D}/raw__{pair_desc(k)}__{slug(name)}.png')

    fname=f'fig09__spatial_eta__rank{rank:02d}__{pair_desc(k)}.png'
    fp=f'{D}/{fname}'
    plt.tight_layout(pad=.4)
    plt.savefig(fp,dpi=230,bbox_inches='tight')
    plt.close(fig)

    saved.append(dict(rank=rank,k=k,file=fp,pair=pair_desc(k)))
    del d,fld
    gc.collect()
    torch.cuda.empty_cache()
    print(rank,'/',len(SPATIAL_KS))

save_index(saved,'fig09_spatial_eta')
print('spatial-eta candidates ->',D)


In [ ]:
# ============================================================
# 11. OPTIONAL FIGURE 10 — 4-WAY ANCHOR-BASED PALETTE CANDIDATES
# ============================================================
RUN_PALETTES=True
N_PALETTES=7
GRID_N=5
D=figdir('fig10_palette')

def endpoint_image(ref):
    k,side=ref
    return load_img(side,k,256)

def endpoint_name(ref):
    k,side=ref
    return f'k{k:04d}-{side}-{endpoint_desc(k,side)}'

def diverse_quartets(nsets=7):
    if ANC is None:
        rr=np.random.RandomState(77)
        return [
            [(int(k),'A') for k in rr.choice(ks,4,replace=False)]
            for _ in range(nsets)
        ]

    refs=[]
    feats=[]
    for k in ks:
        for side in ['A','B']:
            key=f'{k}_{side}'
            if key in ANC:
                refs.append((int(k),side))
                feats.append(ANC[key])
    feats=np.asarray(feats)

    out=[]
    starts=np.linspace(0,len(refs)-1,nsets,dtype=int)
    for st in starts:
        chosen=[int(st)]
        chosen_ks={refs[st][0]}
        while len(chosen)<4:
            dmin=np.full(len(refs),np.inf)
            for q in chosen:
                d=np.linalg.norm(feats-feats[q],axis=1)
                dmin=np.minimum(dmin,d)
            for i,r in enumerate(refs):
                if r[0] in chosen_ks:
                    dmin[i]=-np.inf
            nxt=int(np.argmax(dmin))
            chosen.append(nxt)
            chosen_ks.add(refs[nxt][0])
        out.append([refs[i] for i in chosen])
    return out

if RUN_PALETTES:
    quartets=diverse_quartets(N_PALETTES)
    saved=[]

    for pi,refs in enumerate(quartets,1):
        imgs=[endpoint_image(r) for r in refs]
        anchor=imgs[0]

        fields=[]
        d_keep=None
        for j in range(1,4):
            d=build_d(cnt_load(anchor),cnt_load(imgs[j]))
            fld=transport_field(d,seed=1000*pi+j,eps=EPS)
            if d_keep is None:
                d_keep=d
                anchor_sf=fld['sf']
                shape_info=(fld['L'],fld['H'],fld['W'],fld['Cc'])
            fields.append(fld['target_full'])
            if j>1:
                del d
            del fld

        F1=anchor_sf
        F2,F3,F4=fields
        L,H,W,Cc=shape_info

        tile_dir=f'{D}/tiles__palette{pi:02d}'
        os.makedirs(tile_dir,exist_ok=True)

        tiles=[]
        us=np.linspace(0,1,GRID_N)
        vs=np.linspace(0,1,GRID_N)
        for iy,v in enumerate(vs):
            row=[]
            for ix,u in enumerate(us):
                w1=(1-u)*(1-v)
                w2=u*(1-v)
                w3=(1-u)*v
                w4=u*v
                mix=w1*F1+w2*F2+w3*F3+w4*F4
                im=to_pil(_dec(d_keep,mix,L,H,W,Cc))
                im.save(f'{tile_dir}/v{iy:02d}_u{ix:02d}__w_{w1:.2f}_{w2:.2f}_{w3:.2f}_{w4:.2f}.png')
                row.append(im)
            tiles.append(row)

        fig=plt.figure(figsize=(10.5,8.7),dpi=150)
        gs=fig.add_gridspec(2,5,height_ratios=[1,5])

        for j,(im,r) in enumerate(zip(imgs,refs)):
            ax=fig.add_subplot(gs[0,j])
            ax.imshow(im)
            ax.axis('off')
            ax.set_title(f'T{j+1}: {endpoint_name(r)}',fontsize=7)

        sub=gs[1,:].subgridspec(GRID_N,GRID_N,wspace=.02,hspace=.02)
        for iy in range(GRID_N):
            for ix in range(GRID_N):
                ax=fig.add_subplot(sub[iy,ix])
                ax.imshow(tiles[iy][ix])
                ax.axis('off')

        refslug='__'.join(endpoint_name(r) for r in refs)
        fname=f'fig10__palette__set{pi:02d}__{slug(refslug,120)}.png'
        fp=f'{D}/{fname}'
        plt.tight_layout(pad=.4)
        plt.savefig(fp,dpi=220,bbox_inches='tight')
        plt.close(fig)

        saved.append(dict(set=pi,file=fp,sources=' | '.join(endpoint_name(r) for r in refs)))
        del d_keep,fields,tiles
        gc.collect()
        torch.cuda.empty_cache()
        print('palette',pi,'/',len(quartets))

    save_index(saved,'fig10_palette')
    print('palette candidates ->',D)
else:
    print('RUN_PALETTES=False; skipped')


In [ ]:
# ============================================================
# 12. CONTACT SHEETS — quick visual browsing
# ============================================================
def is_raw_or_contact(path):
    n=os.path.basename(path)
    return n.startswith('raw__') or n.startswith('__CONTACT')

def make_contacts(folder, per_sheet=8, target_w=1500):
    files=[
        f for f in sorted(glob.glob(f'{folder}/*.png'))
        if not is_raw_or_contact(f) and '/tiles__' not in f
    ]
    if not files:
        return []

    outs=[]
    for si in range(0,len(files),per_sheet):
        chunk=files[si:si+per_sheet]
        cards=[]
        for f in chunk:
            im=Image.open(f).convert('RGB')
            scale=min(1.0,target_w/im.width)
            im=im.resize((int(im.width*scale),int(im.height*scale)),Image.Resampling.LANCZOS)

            band=52
            card=Image.new('RGB',(max(target_w,im.width),im.height+band),'white')
            x=(card.width-im.width)//2
            card.paste(im,(x,band))
            dr=ImageDraw.Draw(card)
            dr.text((10,10),os.path.basename(f)[:150],fill='black')
            cards.append(card)

        W=max(c.width for c in cards)
        H=sum(c.height for c in cards)+8*(len(cards)-1)
        sheet=Image.new('RGB',(W,H),'white')
        y=0
        for c in cards:
            sheet.paste(c,((W-c.width)//2,y))
            y+=c.height+8

        out=f'{folder}/__CONTACT_{si//per_sheet+1:02d}.jpg'
        sheet.save(out,quality=88)
        outs.append(out)
    return outs

folders=[
    figdir('fig01_teaser'),
    figdir('fig02_method'),
    figdir('fig03_epsilon'),
    figdir('fig05_main_grid'),
    figdir('fig06_element_zoom'),
    figdir('fig07_boundary'),
    figdir('fig08_transfer'),
    figdir('fig09_spatial_eta'),
]
if RUN_PALETTES:
    folders.append(figdir('fig10_palette'))

for f in folders:
    outs=make_contacts(f)
    print(os.path.basename(f),':',len(outs),'contact sheets')

print('\nDONE. Browse:',PF)
print('Candidate metadata:',IDX)


## Suggested order for today

1. Run **0–3** first and inspect `fig05_main_grid` and `fig06_element_zoom`.
2. Run CNT setup.
3. Run **Figure 1 teaser** candidates.
4. Run **Figure 2 method** candidates.
5. Run **Figure 7 boundary** candidates.
6. Run **Figure 8 transfer** candidates.
7. Run epsilon/spatial/palette only after the core figures have strong examples.

When you have shortlisted candidate filenames, send me the contact sheets or the candidate images and I can help choose the strongest final rows.